# Anomaly Detector Training Notebook\n
\n
This notebook trains a sequence model (e.g. LSTM autoencoder) to detect unusual flight patterns for the Agentic Airspace Monitoring System.\n
\n
It is designed to mirror the preprocessing used in `agents.predict_node` so that runtime inference can share the same normalization statistics.

In [ ]:
# Step 1: Prepare dataset from archive (Fusion_Data + GPS logs)
# Run this once before training. Creates data_prepared/anomaly_windows.npy
import sys
sys.path.insert(0, str(BASE_DIR))
from tools.prepare_anomaly_dataset import main
main()

In [ ]:
import os\n
import math\n
from pathlib import Path\n
\n
import torch\n
import torch.nn as nn\n
from torch.utils.data import Dataset, DataLoader\n
import numpy as np\n
\n
BASE_DIR = Path('..').resolve()\n
DATA_DIR = BASE_DIR / 'data_prepared'\n
\n
SEQ_LEN = 20  # timesteps per window\n
BATCH_SIZE = 64\n
EPOCHS = 10  # tune as needed\n
\n
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
class TrajectoryWindowDataset(Dataset):\n
    \"\"\"\n
    Loads fixed-length windows of (d_lat_m, d_lon_m, d_alt_m) similar to agents.predict_node.\n
    This is intentionally generic; adapt loading to your actual prepared files.\n
    \"\"\"\n
    def __init__(self, npy_path: Path):\n
        arr = np.load(npy_path)  # shape: [N, SEQ_LEN, 3]\n
        assert arr.ndim == 3 and arr.shape[2] == 3\n
        self.data = arr.astype('float32')\n
\n
        # Compute normalization statistics\n
        flat = self.data.reshape(-1, 3)\n
        self.lat_mean, self.lon_mean, self.alt_mean = flat.mean(axis=0)\n
        self.lat_std, self.lon_std, self.alt_std = flat.std(axis=0) + 1e-6\n
\n
        self.data[..., 0] = (self.data[..., 0] - self.lat_mean) / self.lat_std\n
        self.data[..., 1] = (self.data[..., 1] - self.lon_mean) / self.lon_std\n
        self.data[..., 2] = (self.data[..., 2] - self.alt_mean) / self.alt_std\n
\n
    def __len__(self):\n
        return self.data.shape[0]\n
\n
    def __getitem__(self, idx):\n
        return self.data[idx]

In [ ]:
class AnomalyAutoencoder(nn.Module):\n
    def __init__(self, input_size=3, hidden_size=64, latent_size=32):\n
        super().__init__()\n
        self.encoder = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True)\n
        self.to_latent = nn.Linear(hidden_size, latent_size)\n
        self.from_latent = nn.Linear(latent_size, hidden_size)\n
        self.decoder = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True)\n
        self.out = nn.Linear(hidden_size, input_size)\n
\n
    def forward(self, x):\n
        enc_out, _ = self.encoder(x)\n
        h_last = enc_out[:, -1, :]\n
        z = self.to_latent(h_last)\n
        h0 = self.from_latent(z).unsqueeze(0).repeat(2, 1, 1)\n
        c0 = torch.zeros_like(h0)\n
        dec_out, _ = self.decoder(x, (h0, c0))\n
        return self.out(dec_out)

In [ ]:
def train():\n
    npy_path = DATA_DIR / 'anomaly_windows.npy'  # prepare this file with your pipeline\n
    dataset = TrajectoryWindowDataset(npy_path)\n
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)\n
\n
    model = AnomalyAutoencoder().to(DEVICE)\n
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)\n
    crit = nn.MSELoss()\n
\n
    for epoch in range(EPOCHS):\n
        model.train()\n
        total_loss = 0.0\n
        for batch in loader:\n
            batch = batch.to(DEVICE)\n
            opt.zero_grad()\n
            out = model(batch)\n
            loss = crit(out, batch)\n
            loss.backward()\n
            opt.step()\n
            total_loss += loss.item() * batch.size(0)\n
        print(f'Epoch {epoch+1}: loss={total_loss/len(dataset):.6f}')\n
\n
    # Save checkpoint with normalization for runtime use in agents.anomaly_node\n
    ckpt = {\n
        'model_state_dict': model.state_dict(),\n
        'normalization': {\n
            'lat_mean': float(dataset.lat_mean),\n
            'lat_std': float(dataset.lat_std),\n
            'lon_mean': float(dataset.lon_mean),\n
            'lon_std': float(dataset.lon_std),\n
            'alt_mean': float(dataset.alt_mean),\n
            'alt_std': float(dataset.alt_std),\n
        },\n
        'seq_len': SEQ_LEN,\n
    }\n
    out_path = DATA_DIR / 'anomaly_lstm.pth'\n
    torch.save(ckpt, out_path)\n
    print('Saved checkpoint to', out_path)

In [ ]:
# Step 3: Run training
train()